# Experiment Results Analysis 

The object of this notebook is to conduct the final experiment analysis in order to test hypothesis regarding algorithm performance.

In this notebook the different executed batches of experiments will be consolidated in a unique database to analyze its results.


## Analysis Conducted

1. Compare AUC for each algorithm in each enviroment (Comparisons are conducted across algorithms in the same environment)

2. Compare the Mean Best So Far Curve in each enviroment (Comparisons are conducted across algorithms in the same environment) with confidence intervals

3. Perform the wilcoxon Rank Sum test in each enviroment for selected pairwise comaprisons (Comparisons are conducted across algorithms in the same environment) with confidence intervals


## Algorithms to compare

1. Binary Particle Swarm Optimization - (PSO)

2. Binary Global Random Search - (Random)

3. Standard Binary Representation GA (Generic)

4. Standard Mixed Integer Representation GA (mixed_generic)

5. Macro_Micro CX Operator GA - (macro_micro)

6. Macro CX Operator GA - Deactivates Micro CX operator - (micro)

7. Micro CX Operator GA - Deactivates Macro CX operator - (macro)

8. Parents CX Operator - Uses a parent recombination CX Operator (recomb)

## 1. Library Imports

In [1]:
import pandas as pd 
import numpy as np
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt
import seaborn as sns
import os 
import ast
from typing import List, Dict 
import re

## 2. Data Import

In [2]:
# Check wd
print(os.getcwd())



c:\Users\andre\Repositories\FTZ_model_2.0\final_results\experiment


In [3]:
def load_and_concat_csv(paths: List[str], *, ignore_index: bool = True) -> pd.DataFrame:
    """
    Load multiple CSV files and concatenate them into a single DataFrame.

    Parameters
    ----------
    paths : List[str]
        List of file paths to CSV files.
    ignore_index : bool, optional
        If True, reset the index in the concatenated DataFrame. Default is True.

    Returns
    -------
    pd.DataFrame
        Concatenated DataFrame containing all CSV files.
    """
    dataframes = []
    for path in paths:
        try:
            df = pd.read_csv(path)
            df["source_file"] = path  # optional: keep track of origin
            dataframes.append(df)
            print(f"Loaded: {path} ({len(df)} rows)")
        except FileNotFoundError:
            print(f" File not found: {path}")
        except pd.errors.EmptyDataError:
            print(f" Empty file skipped: {path}")
        except Exception as e:
            print(f" Error reading {path}: {e}")

    if not dataframes:
        raise ValueError("No valid CSV files loaded.")

    combined_df = pd.concat(dataframes, ignore_index=ignore_index)
    print(f" Combined DataFrame: {combined_df.shape[0]} rows, {combined_df.shape[1]} columns")

    return combined_df

In [ ]:
paths = [
    r"test_suite\data_exp\experiment_final_macrmicro_mixedgeneric\experiment_final_macrmicro_mixedgeneric\results.csv",
    r"test_suite\data_exp\experiment_final_nocross_jmicro_jmacro_recomb\experiment_final_nocross_jmicro_jmacro_recomb\results.csv",
    r"test_suite\data_exp\final_experiment_rand_pso_gen (1)\final_experiment_rand_pso_gen\results.csv"
]


In [5]:
df_all = load_and_concat_csv(paths)

 File not found: data_exp\experiment_final_macrmicro_mixedgeneric\experiment_final_macrmicro_mixedgeneric\results.csv
 File not found: data_exp\experiment_final_nocross_jmicro_jmacro_recomb\experiment_final_nocross_jmicro_jmacro_recomb\results.csv
 File not found: data_exp\final_experiment_rand_pso_gen (1)\final_experiment_rand_pso_gen\results.csv


ValueError: No valid CSV files loaded.

## 3. Clean Data 

Due to a code / paralleization mistake, some rows are duplicated

In [ ]:
df_all.head()

,env_name,run,algorithm,seed,best_curve,best_value,best_genome,all_best_genomes,elapsed_sec,success,source_file
0,Linear_standard,1,macro_micro,483575,"[38.9, 39.05, 39.05, 39.2, 39.35, 39.35, 39.35...",39.35,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",81.426382,True,data_exp\experiment_final_macrmicro_mixedgener...
1,Linear_standard,11,macro_micro,1424007,"[38.75, 38.9, 38.9, 39.05, 39.05, 39.2, 39.2, ...",39.35,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",81.670461,True,data_exp\experiment_final_macrmicro_mixedgener...
2,Linear_standard,13,macro_micro,1242588,"[38.75, 38.9, 39.05, 39.05, 39.2, 39.2, 39.2, ...",39.35,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",81.777966,True,data_exp\experiment_final_macrmicro_mixedgener...
3,Linear_standard,10,macro_micro,874627,"[39.05, 39.05, 39.05, 39.2, 39.2, 39.2, 39.35,...",39.35,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",81.887681,True,data_exp\experiment_final_macrmicro_mixedgener...
4,Linear_standard,6,macro_micro,1081112,"[38.75, 38.9, 39.05, 39.05, 39.35, 39.35, 39.3...",39.35,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",82.319809,True,data_exp\experiment_final_macrmicro_mixedgener...


In [ ]:
df_all.columns

Index(['env_name', 'run', 'algorithm', 'seed', 'best_curve', 'best_value',
       'best_genome', 'all_best_genomes', 'elapsed_sec', 'success',
       'source_file'],
      dtype='object')

In [ ]:
# Keep the row with the smallest elapsed_sec per (env_name, algorithm, seed, run)
before = len(df_all)
print(f"Total rows before removing duplicates: {before}")

df_tmp = df_all.copy()
df_tmp["elapsed_sec"] = pd.to_numeric(df_tmp.get("elapsed_sec", np.nan), errors="coerce")
df_tmp["_elapsed_sort"] = df_tmp["elapsed_sec"].fillna(np.inf)

df_analysis = (
    df_tmp
    .sort_values(["env_name", "algorithm", "seed", "run", "_elapsed_sort"],
                 ascending=[True, True, True, True, True])
    .drop_duplicates(subset=["env_name", "algorithm", "seed", "run"], keep="first")
    .drop(columns=["_elapsed_sort"])
    .reset_index(drop=True)
)

after = len(df_analysis)
print(f"Removed {before - after} duplicate rows")


Total rows before removing duplicates: 146160
Removed 136080 duplicate rows


In [ ]:
df_analysis.shape

(10080, 11)

In [ ]:
df_analysis.head()

,env_name,run,algorithm,seed,best_curve,best_value,best_genome,all_best_genomes,elapsed_sec,success,source_file
0,All Inputs_perturbed,1,generic,488498,"[353.55, 355.5, 355.5, 355.5, 355.8, 355.8, 35...",357.45,"[1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1]",[],68.275849,True,data_exp\final_experiment_rand_pso_gen (1)\fin...
1,All Inputs_perturbed,17,generic,614685,"[352.5, 353.1, 354.3, 354.3, 354.6, 355.2, 355...",357.45,"[1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1]",[],74.619402,True,data_exp\final_experiment_rand_pso_gen (1)\fin...
2,All Inputs_perturbed,24,generic,638677,"[351.45, 351.45, 353.1, 353.1, 354.15, 354.15,...",357.45,"[1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1]",[],66.299484,True,data_exp\final_experiment_rand_pso_gen (1)\fin...
3,All Inputs_perturbed,2,generic,639652,"[354.3, 354.3, 356.4, 356.4, 356.7, 356.7, 356...",357.45,"[1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1]",[],70.683193,True,data_exp\final_experiment_rand_pso_gen (1)\fin...
4,All Inputs_perturbed,9,generic,646853,"[352.35, 354.6, 355.95, 356.25, 356.55, 357.15...",357.45,"[1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1]",[],79.252405,True,data_exp\final_experiment_rand_pso_gen (1)\fin...


## 4. Graphics of Mean Best So Far Curves

In [ ]:
df_graphics = df_analysis[["env_name", "run", "seed", "algorithm", "best_curve", "best_value"]].copy()


In [ ]:
df_graphics.head()

,env_name,run,seed,algorithm,best_curve,best_value
0,All Inputs_perturbed,1,488498,generic,"[353.55, 355.5, 355.5, 355.5, 355.8, 355.8, 35...",357.45
1,All Inputs_perturbed,17,614685,generic,"[352.5, 353.1, 354.3, 354.3, 354.6, 355.2, 355...",357.45
2,All Inputs_perturbed,24,638677,generic,"[351.45, 351.45, 353.1, 353.1, 354.15, 354.15,...",357.45
3,All Inputs_perturbed,2,639652,generic,"[354.3, 354.3, 356.4, 356.4, 356.7, 356.7, 356...",357.45
4,All Inputs_perturbed,9,646853,generic,"[352.35, 354.6, 355.95, 356.25, 356.55, 357.15...",357.45


In [ ]:
df_graphics["best_curve"].dtype


dtype('O')

In [ ]:
def safe_parse_with_inf(x):
    if not isinstance(x, str):
        return np.array(x, dtype=float) if isinstance(x, (list, np.ndarray)) else np.array([])

    # Replace 'inf', '-inf', 'nan' with strings that eval can interpret
    x = re.sub(r'\b-inf\b', 'float("-inf")', x)
    x = re.sub(r'\binf\b', 'float("inf")', x)
    x = re.sub(r'\bnan\b', 'float("nan")', x)

    try:
        parsed = ast.literal_eval(x)
        return np.array(parsed, dtype=float)
    except Exception:
        return np.array([])

df_graphics["best_curve"] = df_graphics["best_curve"].apply(safe_parse_with_inf)

In [ ]:
df_graphics.columns

Index(['env_name', 'run', 'seed', 'algorithm', 'best_curve', 'best_value'], dtype='object')

### Graphics 

In [ ]:
def clean_curve(arr):
    arr = np.array(arr, dtype=float)
    mask_invalid = ~np.isfinite(arr)  # detects inf, -inf, nan
    n_invalid = np.sum(mask_invalid)
    arr = arr[np.isfinite(arr)]       # keep only finite values
    return arr, n_invalid

# Clean curves and count removals
invalid_counts = []
cleaned_curves = []

for idx, arr in enumerate(df_graphics["best_curve"]):
    cleaned, n_invalid = clean_curve(arr)
    invalid_counts.append(n_invalid)
    cleaned_curves.append(cleaned)

df_graphics["best_curve"] = cleaned_curves
df_graphics["invalid_points"] = invalid_counts

print(f"Total invalid values removed: {sum(invalid_counts)}")

Total invalid values removed: 0


In [ ]:
from scipy.stats import t

from scipy.stats import t
import numpy as np

def mean_curve_with_ci(curves, debug=False):
    """
    Computes the mean curve and 95% confidence interval across runs.
    Filters out invalid, empty, or malformed curves.
    """

    clean_curves = []
    for i, c in enumerate(curves):
        # Skip non-list/array items or empty ones
        if not isinstance(c, (list, np.ndarray)):
            if debug:
                print(f"⚠️ Run {i} skipped (not list/array): {type(c)} -> {c}")
            continue
        c = np.array(c, dtype=float).flatten()

        # Skip empty or NaN-only curves
        if len(c) == 0 or not np.isfinite(np.sum(c)):
            if debug:
                print(f"⚠️ Run {i} skipped (empty or invalid values): {c}")
            continue

        clean_curves.append(c)

    # If none valid, return empty arrays
    if len(clean_curves) == 0:
        if debug:
            print("❌ No valid curves in group.")
        return np.array([]), np.array([]), np.array([])

    # Truncate to the shortest length to align
    min_len = min(len(c) for c in clean_curves)
    if debug:
        print(f"✅ Using {len(clean_curves)} valid curves, truncated to length {min_len}")

    data = np.stack([c[:min_len] for c in clean_curves], axis=0)

    # Compute statistics
    mean = np.nanmean(data, axis=0)
    std = np.nanstd(data, axis=0, ddof=1)
    n = data.shape[0]
    alpha = 0.05
    tcrit = t.ppf(1 - alpha/2, df=n - 1) if n > 1 else 0
    se = std / np.sqrt(n)
    ci = tcrit * se

    return mean, mean - ci, mean + ci


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# === Output folder on your Desktop ===
desktop = os.path.join(os.path.expanduser("~"), "Desktop")
output_dir = os.path.join(desktop, "FTZ_Plots")
os.makedirs(output_dir, exist_ok=True)
print(f" Plots will be saved in: {output_dir}\n")

# === Group by environment and algorithm ===
grouped = df_graphics.groupby(["env_name", "algorithm"])

# Unique environments
envs = df_graphics["env_name"].unique()

#  Print how many and which environments will be plotted
print(f" Plotting {len(envs)} environments:")
for i, e in enumerate(envs, 1):
    print(f"   {i}. {e}")




 Plots will be saved in: C:\Users\andre\Desktop\FTZ_Plots

 Plotting 28 environments:
   1. All Inputs_perturbed
   2. All Inputs_standard
   3. Assembly Tree_perturbed
   4. Assembly Tree_standard
   5. Butterfly_perturbed
   6. Butterfly_standard
   7. Dominant Input_perturbed
   8. Dominant Input_standard
   9. Explosive Expansion_perturbed
   10. Explosive Expansion_standard
   11. Interlaced Pathway_perturbed
   12. Interlaced Pathway_standard
   13. Linear All-Inputs_perturbed
   14. Linear All-Inputs_standard
   15. Linear_perturbed
   16. Linear_standard
   17. Mangrove_perturbed
   18. Mangrove_standard
   19. Mixed Test Graph_perturbed
   20. Mixed Test Graph_standard
   21. Sequential Mergers with External Inputs_perturbed
   22. Sequential Mergers with External Inputs_standard
   23. Successive Diamonds_perturbed
   24. Successive Diamonds_standard
   25. Tri-Cluster_perturbed
   26. Tri-Cluster_standard
   27. Tri-Spine Convergent_perturbed
   28. Tri-Spine Convergent_stan

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# === Configuración global de estilo ===
plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "legend.fontsize": 12,
    "lines.linewidth": 2,
    "lines.markersize": 6,
    "axes.linewidth": 1.2,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "figure.dpi": 300,
    "savefig.dpi": 400,
    "figure.autolayout": True
})

# === Selección de algoritmos a graficar ===
# Deja la lista vacía [] para graficar todos los algoritmos del DataFrame
algorithms_to_plot = []  # ejemplo
# algorithms_to_plot = []  # -> todos los algoritmos si se deja vacío

# Si está vacía, usar todos los algoritmos disponibles en el df
if not algorithms_to_plot:
    algorithms_to_plot = df_graphics["algorithm"].unique().tolist()

print(f"📊 Se graficarán {len(algorithms_to_plot)} algoritmos: {algorithms_to_plot}")

# === Plot por entorno ===
for env in envs:
    subdf_env = df_graphics[df_graphics["env_name"] == env]
    fig, ax = plt.subplots(figsize=(10, 6))

    # Filtrar solo los algoritmos deseados
    subdf_env = subdf_env[subdf_env["algorithm"].isin(algorithms_to_plot)]

    for algo, df_algo in subdf_env.groupby("algorithm"):
        curves = df_algo["best_curve"].tolist()
        mean, lower, upper = mean_curve_with_ci(curves, debug=False)
        if len(mean) == 0:
            continue

        x = np.arange(len(mean))
        ax.plot(x, mean, label=algo, alpha=0.9)
        ax.fill_between(x, lower, upper, alpha=0.2)

    # === Ajustes estéticos ===
    ax.set_title(f"Environment:{env}", fontweight="bold", pad=12)
    ax.set_xlabel("Generation", labelpad=10)
    ax.set_ylabel("Mean Best So Far Curve", labelpad=10)
    ax.legend(frameon=True, loc="best", fancybox=True, shadow=False, ncol=2)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(True, linestyle="--", alpha=0.5)

    # === Guardar ===
    safe_name = "".join(c if c.isalnum() or c in ('_', '-') else "_" for c in env)
    png_path = os.path.join(output_dir, f"{safe_name}.png")
    plt.savefig(png_path, bbox_inches="tight", transparent=False)
    print(f"✅ Saved: {png_path}")
    plt.close(fig)

print("\n All environment plots saved successfully!")



📊 Se graficarán 9 algoritmos: ['generic', 'macro', 'macro_micro', 'micro', 'mixed_generic', 'no_crossover', 'pso', 'random', 'recomb']
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\All_Inputs_perturbed.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\All_Inputs_standard.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Assembly_Tree_perturbed.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Assembly_Tree_standard.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Butterfly_perturbed.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Butterfly_standard.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Dominant_Input_perturbed.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Dominant_Input_standard.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Explosive_Expansion_perturbed.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Explosive_Expansion_standard.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Interlaced_Pathway_perturbed.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Interlaced_Pathway_standard.png
✅ Saved: C:\Users\and

In [ ]:
df_graphics['env_name'].unique()

array(['All Inputs_perturbed', 'All Inputs_standard',
       'Assembly Tree_perturbed', 'Assembly Tree_standard',
       'Butterfly_perturbed', 'Butterfly_standard',
       'Dominant Input_perturbed', 'Dominant Input_standard',
       'Explosive Expansion_perturbed', 'Explosive Expansion_standard',
       'Interlaced Pathway_perturbed', 'Interlaced Pathway_standard',
       'Linear All-Inputs_perturbed', 'Linear All-Inputs_standard',
       'Linear_perturbed', 'Linear_standard', 'Mangrove_perturbed',
       'Mangrove_standard', 'Mixed Test Graph_perturbed',
       'Mixed Test Graph_standard',
       'Sequential Mergers with External Inputs_perturbed',
       'Sequential Mergers with External Inputs_standard',
       'Successive Diamonds_perturbed', 'Successive Diamonds_standard',
       'Tri-Cluster_perturbed', 'Tri-Cluster_standard',
       'Tri-Spine Convergent_perturbed', 'Tri-Spine Convergent_standard'],
      dtype=object)

## 5. Hypothesis testing

In [ ]:
df_wilcoxon = df_graphics[["env_name", "algorithm", "best_value"]].copy()

df_wilcoxon.head()

,env_name,algorithm,best_value
0,All Inputs_perturbed,generic,357.45
1,All Inputs_perturbed,generic,357.45
2,All Inputs_perturbed,generic,357.45
3,All Inputs_perturbed,generic,357.45
4,All Inputs_perturbed,generic,357.45


In [ ]:
pairs = [
    # macro_micro vs others
    ("macro_micro", "mixed_generic"),
    ("mixed_generic", "macro_micro"),
    ("macro_micro", "generic"),
    ("generic", "macro_micro"),
    ("macro_micro", "macro"),
    ("macro", "macro_micro"),
    ("macro_micro", "micro"),
    ("micro", "macro_micro"),
    ('recomb','macro_micro'),
    ('macro_micro','recomb'),

    # mixed_generic vs others
    ("mixed_generic", "pso"),
    ("pso", "mixed_generic"),
    ("mixed_generic", "generic"),
    ("generic", "mixed_generic"),

    # pso vs others
    ("pso", "random"),
    ("random", "pso"),
]


alpha = 0.05  # significance level
results = []


In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import pandas as pd

alpha = 0.05
results = []

print(f"\n Performing pairwise Mann–Whitney U tests at α={alpha}\n")

# Loop over each environment
for env in df_wilcoxon["env_name"].unique():
    df_env = df_wilcoxon[df_wilcoxon["env_name"] == env]

    for a1, a2 in pairs:
        # Ensure both algorithms exist in this environment
        if a1 not in df_env["algorithm"].unique() or a2 not in df_env["algorithm"].unique():
            continue

        vals1 = df_env.loc[df_env["algorithm"] == a1, "best_value"].astype(float).values
        vals2 = df_env.loc[df_env["algorithm"] == a2, "best_value"].astype(float).values

        # Skip if too few observations
        if len(vals1) < 3 or len(vals2) < 3:
            continue

        # Mann–Whitney test (one-sided): H₁ = a1 > a2
        stat, p = mannwhitneyu(vals1, vals2, alternative="greater")

        results.append({
            "env_name": env,
            "algorithm_1": a1,
            "algorithm_2": a2,
            "p_value": p
        })

# Create DataFrame of all tests
df_mannwhitney = pd.DataFrame(results)

# Uncorrected decisions
df_mannwhitney["reject_uncorrected"] = df_mannwhitney["p_value"] < alpha

# === Apply Bonferroni correction globally across all tests ===
reject_bonf, p_adj_bonf, _, _ = multipletests(df_mannwhitney["p_value"], alpha=alpha, method="bonferroni")
df_mannwhitney["p_value_bonferroni"] = p_adj_bonf
df_mannwhitney["reject_bonferroni"] = reject_bonf

# === Summaries ===
summary = (
    df_mannwhitney.groupby(["algorithm_1", "algorithm_2"])
    .agg(
        envs_where_a1_better_uncorrected=("reject_uncorrected", "sum"),
        envs_where_a1_better_bonferroni=("reject_bonferroni", "sum")
    )
    .reset_index()
)

# === Print results ===
print("\n Detailed per-environment results:")
print(df_mannwhitney.to_string(index=False))

print("\n Summary across environments:")
print(summary.to_string(index=False))





 Performing pairwise Mann–Whitney U tests at α=0.05


 Detailed per-environment results:
                                         env_name   algorithm_1   algorithm_2      p_value  reject_uncorrected  p_value_bonferroni  reject_bonferroni
                             All Inputs_perturbed   macro_micro mixed_generic 1.000000e+00               False        1.000000e+00              False
                             All Inputs_perturbed mixed_generic   macro_micro 1.000000e+00               False        1.000000e+00              False
                             All Inputs_perturbed   macro_micro       generic 1.000000e+00               False        1.000000e+00              False
                             All Inputs_perturbed       generic   macro_micro 1.000000e+00               False        1.000000e+00              False
                             All Inputs_perturbed   macro_micro         macro 1.000000e+00               False        1.000000e+00              False
          

In [ ]:
# Hello